In [ ]:
"""
Skenario 1: NeuMF Baseline (tanpa Cluster, tanpa TPE)
Arsitektur sengaja lebih lemah dibanding Skenario 2 & 4:
  - MLP hanya 2 layer: [emb_dim*2 → 64 → 32]
  - Tidak ada _init_weights / Xavier init (pakai default PyTorch)
  - Semua hyperparameter fixed, tidak ada tuning
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import math

# =====================
# CONFIG
# =====================
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 30
BATCH_SIZE   = 1024
LR           = 1e-3
EMB_DIM      = 64
DROPOUT      = 0.2
TOP_K        = 10
NUM_NEG      = 99
RANDOM_STATE = 42

print("=" * 50)
print("  Skenario 1: NeuMF Baseline")
print("=" * 50)
print("Device:", DEVICE)

# =====================
# LOAD DATA
# =====================
print("\nMemuat dataset...")
train_df = pd.read_csv("data/train_dataset.csv")
test_df  = pd.read_csv("data/test_dataset.csv")
full_df  = pd.read_csv("data/user_dataset_final.csv")

n_users = full_df['user_id_enc'].max() + 1
n_items = full_df['item_id_enc'].max() + 1
print(f"Users: {n_users}, Items: {n_items}")
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Kumpulkan item positif per user dari train — mencegah negatif bocor saat evaluasi
user_positive_items = (
    train_df[train_df['label'] == 1]
    .groupby('user_id_enc')['item_id_enc']
    .apply(set)
    .to_dict()
)

# =====================
# DATALOADER
# =====================
dataset = TensorDataset(
    torch.tensor(train_df['user_id_enc'].values).long(),
    torch.tensor(train_df['item_id_enc'].values).long(),
    torch.tensor(train_df['label'].values).float()
)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)
print(f"Training samples: {len(dataset)}")

# =====================
# MODEL — SENGAJA LEMAH
# MLP hanya 2 layer, tidak ada Xavier init
# =====================
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, dropout=0.2):
        super().__init__()
        # GMF path
        self.user_gmf = nn.Embedding(n_users, emb_dim)
        self.item_gmf = nn.Embedding(n_items, emb_dim)
        # MLP path
        self.user_mlp = nn.Embedding(n_users, emb_dim)
        self.item_mlp = nn.Embedding(n_items, emb_dim)

        # MLP 2 layer 
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(dropout),
            nn.ReLU()
        )
        # Output: GMF (emb_dim) + MLP last (32)
        self.output = nn.Linear(emb_dim + 32, 1)
        # Tidak ada _init_weights — pakai default PyTorch (Kaiming Uniform)

    def forward(self, user, item):
        gmf    = self.user_gmf(user) * self.item_gmf(item)
        mlp_in = torch.cat([self.user_mlp(user), self.item_mlp(item)], dim=-1)
        mlp    = self.mlp(mlp_in)
        x      = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze()

torch.manual_seed(RANDOM_STATE)
model     = NeuMF(n_users, n_items, emb_dim=EMB_DIM, dropout=DROPOUT).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# =====================
# EVALUASI — LOO + 99 NEGATIF
# =====================
@torch.no_grad()
def evaluate(model, seed=RANDOM_STATE):
    model.eval()
    hits, ndcgs = [], []
    rng = np.random.default_rng(seed)

    # Ambil 1 item positif per user dari test_df
    test_pos = test_df[test_df['label'] == 1].copy()

    for row in tqdm(test_pos.itertuples(index=False), total=len(test_pos),
                    desc="Evaluating", leave=False):
        user      = int(row.user_id_enc)
        true_item = int(row.item_id_enc)
        positives = user_positive_items.get(user, set())

        # Sample 99 negatif yang tidak bocor
        negatives = set()
        while len(negatives) < NUM_NEG:
            j = int(rng.integers(n_items))
            if j != true_item and j not in positives:
                negatives.add(j)

        items_eval = list(negatives) + [true_item]
        users_eval = [user] * len(items_eval)

        user_t = torch.tensor(users_eval).long().to(DEVICE)
        item_t = torch.tensor(items_eval).long().to(DEVICE)

        scores    = torch.sigmoid(model(user_t, item_t)).cpu().numpy()
        rank      = np.argsort(scores)[::-1]
        top_items = np.array(items_eval)[rank[:TOP_K]]

        if true_item in top_items:
            r = int(np.where(top_items == true_item)[0][0]) + 1
            hits.append(1)
            ndcgs.append(1.0 / math.log2(r + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return np.mean(hits), np.mean(ndcgs)

# =====================
# TRAINING
# =====================
print("\nMemulai Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for u, i, l in loader:
        u, i, l = u.to(DEVICE), i.to(DEVICE), l.to(DEVICE)
        pred = model(u, i)
        loss = criterion(pred, l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f}")

# =====================
# EVALUASI MULTI-SEED
# =====================
EVAL_SEEDS = [42, 123, 456]
print("\nEvaluasi Multi-Seed...")
hr_list, ndcg_list = [], []
for seed in EVAL_SEEDS:
    hr, ndcg = evaluate(model, seed=seed)
    hr_list.append(hr)
    ndcg_list.append(ndcg)
    print(f"  Seed {seed:3d} | HR@{TOP_K}={hr:.4f}, NDCG@{TOP_K}={ndcg:.4f}")

mean_hr   = float(np.mean(hr_list))
mean_ndcg = float(np.mean(ndcg_list))
std_hr    = float(np.std(hr_list))
std_ndcg  = float(np.std(ndcg_list))

print(f"\n{'='*50}")
print(f"  HASIL EVALUASI — Skenario 1: NeuMF Baseline")
print(f"{'='*50}")
print(f"  Mean HR@{TOP_K}    : {mean_hr:.4f} ± {std_hr:.4f}")
print(f"  Mean NDCG@{TOP_K}  : {mean_ndcg:.4f} ± {std_ndcg:.4f}")

# =====================
# SIMPAN HASIL
# =====================
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

result_df = pd.DataFrame([{
    "skenario"     : 1,
    "model"        : "NeuMF Baseline",
    "cluster"      : False,
    "tpe"          : False,
    "HR@10_mean"   : round(mean_hr, 4),
    "HR@10_std"    : round(std_hr, 4),
    "NDCG@10_mean" : round(mean_ndcg, 4),
    "NDCG@10_std"  : round(std_ndcg, 4),
    "seeds"        : str(EVAL_SEEDS),
    "emb_dim"      : EMB_DIM,
    "dropout"      : DROPOUT,
    "lr"           : LR,
    "batch_size"   : BATCH_SIZE,
    "epochs"       : EPOCHS,
    "notes"        : "MLP 2 layer, no Xavier init, fixed params"
}])

result_path = OUTPUT_DIR / "hasil_skenario1_baseline.csv"
result_df.to_csv(result_path, index=False)
print(f"\nHasil disimpan: {result_path}")

Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/skenario1_baseline.pt")
print("Model disimpan: models/skenario1_baseline.pt")

  Skenario 1: NeuMF Baseline
Device: cuda

Memuat dataset...
Users: 9529, Items: 4144
Train size: 2758490, Test size: 952900
Training samples: 2758490

Memulai Training...
Epoch 01/30 | Loss: 1114.1512
Epoch 02/30 | Loss: 1023.4580
Epoch 03/30 | Loss: 990.0555
Epoch 04/30 | Loss: 959.2835
Epoch 05/30 | Loss: 923.0693
Epoch 06/30 | Loss: 884.3311
Epoch 07/30 | Loss: 843.4257
Epoch 08/30 | Loss: 802.0496
Epoch 09/30 | Loss: 759.7492
Epoch 10/30 | Loss: 718.6713
Epoch 11/30 | Loss: 678.6414
Epoch 12/30 | Loss: 639.5995
Epoch 13/30 | Loss: 602.2319
Epoch 14/30 | Loss: 566.5929
Epoch 15/30 | Loss: 533.4892
Epoch 16/30 | Loss: 502.2127
Epoch 17/30 | Loss: 473.1559
Epoch 18/30 | Loss: 446.0761
Epoch 19/30 | Loss: 421.0355
Epoch 20/30 | Loss: 398.0566
Epoch 21/30 | Loss: 376.7596
Epoch 22/30 | Loss: 356.6165
Epoch 23/30 | Loss: 338.4507
Epoch 24/30 | Loss: 321.0106
Epoch 25/30 | Loss: 305.2891
Epoch 26/30 | Loss: 290.8962
Epoch 27/30 | Loss: 276.8753
Epoch 28/30 | Loss: 263.9138
Epoch 29/30 | 

  Seed  42 | HR@10=0.7520, NDCG@10=0.4932


  Seed 123 | HR@10=0.7516, NDCG@10=0.4948


  Seed 456 | HR@10=0.7520, NDCG@10=0.4933

  HASIL EVALUASI — Skenario 1: NeuMF Baseline
  Mean HR@10    : 0.7519 ± 0.0002
  Mean NDCG@10  : 0.4937 ± 0.0007

Hasil disimpan: outputs\hasil_skenario1_baseline.csv
Model disimpan: models/skenario1_baseline.pt
